## **3-stimulus / 2-context swap task — RNN actor-critic**

Value matrix (fixed, no noise):

| stimulus | ctx 0 | ctx 1 |
|----------|-------|-------|
| s0       | 0.0   | 1.0   |
| s1       | 0.5   | 0.5   |
| s2       | 1.0   | 0.0   |

s0 and s2 are swapped across contexts; s1 is always 50/50.

Author: patrick.mccarthy@dpag.ox.ac.uk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

from cxval.tasks import StimulusSequence, StateSequence
from cxval.envs import TaskEnv
from cxval.models import RNN, ActorCritic
from cxval.agents import Agent
from cxval.analysis import (
    filter_act_dict, mean_pairs, mean_offdiag,
    pairwise_decode, crosscontext_decode, generalisation_matrix,
    value_decode_within, value_decode_cross, value_gen_matrix,
    plot_generalisation_heatmap,
)
from cxval.vis import STYLE

plt.style.use(STYLE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Task

In [ ]:
# fixed value matrix — no noise
# rows = stimuli (s0, s1, s2), cols = contexts (c0, c1)
value_matrix = np.array([
    [0.0, 1.0],   # s0: 0 in c0, 1 in c1
    [0.5, 0.5],   # s1: always 0.5
    [1.0, 0.0],   # s2: 1 in c0, 0 in c1
], dtype=np.float32)

n_contexts = 2
n_stimuli  = 3
contexts   = ["c0", "c1"]
stimuli    = ["s0", "s1", "s2"]

# task timing
trials_per_phase   = 50
phases_per_context = 1
context_reps       = 10
stim_timesteps     = 5
reward_timesteps   = 3
iti_timesteps      = (3, 8)
seed               = 42

stim_seq = StimulusSequence(
    value_matrix=value_matrix,
    trials_per_phase=trials_per_phase,
    phases_per_context=phases_per_context,
    context_order="sequential",
    context_reps=context_reps,
)
stim_seq.generate(seed=seed)

state_seq = StateSequence(
    stimulus_sequence=stim_seq,
    value_matrix=value_matrix,
    stim_timesteps=stim_timesteps,
    reward_timesteps=reward_timesteps,
    iti_timesteps=iti_timesteps,
)
states, rewards, reward_availability = state_seq.generate(seed=seed)

n_trials = len(stim_seq.trial_contexts)
obs_dim  = states.shape[1] + 2

print(f"Value matrix:\n{value_matrix}")
print(f"Trials: {n_trials}  |  Sequence: {len(states)} timesteps  |  Obs dim: {obs_dim}")

In [ ]:
# value matrix heatmap
fig, ax = plt.subplots(figsize=(3, 3))
im = ax.imshow(value_matrix, cmap="hot", vmin=0, vmax=1, aspect="equal")
for i in range(n_stimuli):
    for j in range(n_contexts):
        v = value_matrix[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                color="white" if v < 0.55 else "black", fontsize=11, fontweight="bold")
ax.set_yticks(range(n_stimuli)); ax.set_yticklabels(stimuli)
ax.set_xticks(range(n_contexts)); ax.set_xticklabels(contexts)
ax.set_ylabel("Stimulus"); ax.set_xlabel("Context")
ax.set_title("Value matrix")
plt.colorbar(im, ax=ax, shrink=0.8, label="Reward probability")
plt.tight_layout()
plt.show()

In [ ]:
# state sequence structure
trial_starts     = np.array([t["trial_start"] for t in state_seq.trial_structure])
context_at_trial = np.array([t["context"]     for t in state_seq.trial_structure])
ctx_change_idx   = np.where(np.diff(context_at_trial) != 0)[0] + 1
context_bnd_ts   = trial_starts[ctx_change_idx]

iti_indicator = (
    (states[:, n_contexts : n_contexts + n_stimuli].sum(axis=1) == 0) &
    (states[:, -1] == 0)
).astype(float)

vis_states = np.vstack([
    states.T,
    reward_availability[None, :],
    iti_indicator[None, :],
])
ylabels = (
    [f"ctx {c}" for c in contexts]
    + [f"stim {s}" for s in stimuli]
    + ["window_cue", "reward_avail", "ITI"]
)

ctx_colors = [plt.cm.Set2(v) for v in np.linspace(0, 0.75, max(n_contexts, 1))]

fig, ax = plt.subplots(figsize=(18, 4))
im = ax.imshow(vis_states, aspect="auto", cmap="hot", vmin=0, vmax=1, interpolation="nearest")
for t in context_bnd_ts:
    ax.axvline(t - 0.5, color="white", linewidth=2.5)
ax.set_yticks(range(len(ylabels)))
ax.set_yticklabels(ylabels)
ax.set_xlabel("Timestep")
ax.set_title("State sequence")
plt.colorbar(im, ax=ax, label="Activation")
plt.tight_layout()
plt.show()

## 2. Model

In [ ]:
lick_cost    = 0.0    # penalty for licking when no reward available, any timestep (0 = off)
policy_clip  = 0.05   # min probability for any action; prevents collapse to 0 (0 = off)

env = TaskEnv(
    states=states,
    reward_availability=reward_availability,
    reward_lick=1.0,
    reward_no_lick=0.0,
    reward_lick_miss=-1.0,
    lick_cost=lick_cost,
)

hidden_size    = 64
num_actions    = 2
recurrent_gain = 0.9

backbone     = RNN(input_size=obs_dim, hidden_size=hidden_size, output_size=1,
                   recurrent_gain=recurrent_gain)
actor_critic = ActorCritic(backbone=backbone, num_actions=num_actions,
                           policy_clip=policy_clip).to(device)
agent        = Agent(actor_critic, device=device)

n_params = sum(p.numel() for p in actor_critic.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print(f"policy_clip={policy_clip}  lick_cost={lick_cost}")

## 3. Train

In [ ]:
def compute_returns(rewards, bootstrap_value, gamma):
    returns, R = [], float(bootstrap_value)
    for r in reversed(rewards):
        R = r + gamma * R
        returns.append(R)
    returns.reverse()
    return returns

In [ ]:
n_episodes   = 1
update_every = 200
gamma        = 0.9
lr           = 3e-4
value_coef   = 0.5
entropy_coef = 0.01
grad_clip    = 1.0

optimizer = torch.optim.Adam(actor_critic.parameters(), lr=lr)

actor_losses  = []
critic_losses = []
all_trial_data = []
hidden_np_train = None

for episode in range(n_episodes):
    obs, _ = env.reset()
    hidden = None
    actor_critic.train()

    log_probs_buf, values_buf, rewards_buf, entropies_buf = [], [], [], []
    action_list, reward_list, info_list = [], [], []
    value_ts = []

    is_last = (episode == n_episodes - 1)
    if is_last:
        hidden_ts_list = []

    done = False
    t    = 0

    while not done:
        obs_t  = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        logits, value, hidden = actor_critic.step(obs_t, hidden)
        dist   = actor_critic.make_dist(logits)
        action = dist.sample()

        log_probs_buf.append(dist.log_prob(action))
        values_buf.append(value)
        entropies_buf.append(dist.entropy())
        value_ts.append(value.detach().item())

        if is_last:
            hidden_ts_list.append(hidden.detach().squeeze(0))

        action_list.append(action.item())
        obs, reward, done, _, info = env.step(action.item())
        rewards_buf.append(reward)
        reward_list.append(reward)
        info_list.append(info)
        t += 1

        if t % update_every == 0 or done:
            if done:
                bootstrap_v = 0.0
            else:
                with torch.no_grad():
                    obs_next = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                    _, bv, _ = actor_critic.step(obs_next, hidden)
                    bootstrap_v = bv.item()

            returns      = compute_returns(rewards_buf, bootstrap_v, gamma)
            log_probs_t  = torch.stack(log_probs_buf).squeeze(-1)
            values_t     = torch.stack(values_buf).squeeze(-1)
            returns_t    = torch.tensor(returns, dtype=torch.float32, device=device)
            entropy_mean = torch.stack(entropies_buf).mean()

            advantages = returns_t - values_t.detach()
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

            actor_loss  = -(log_probs_t * advantages).mean()
            critic_loss = F.mse_loss(values_t, returns_t)
            loss        = actor_loss + value_coef * critic_loss - entropy_coef * entropy_mean

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(actor_critic.parameters(), grad_clip)
            optimizer.step()

            hidden = hidden.detach()
            actor_losses.append(actor_loss.item())
            critic_losses.append(critic_loss.item())

            log_probs_buf, values_buf, rewards_buf, entropies_buf = [], [], [], []

    if is_last:
        hidden_np_train = np.array(torch.stack(hidden_ts_list).tolist(), dtype=np.float32)

    for ti, trial in enumerate(state_seq.trial_structure):
        rs, re = trial["reward_window"]
        all_trial_data.append({
            "global_trial":     episode * n_trials + ti,
            "context":          trial["context"],
            "stimulus":         trial["stimulus"],
            "reward_available": trial["reward_available"],
            "licked":           int(action_list[rs] == TaskEnv.LICK),
            "value_estimate":   float(np.mean(value_ts[rs:re])),
            "lick_count":       sum(1 for a in action_list[rs:re] if a == TaskEnv.LICK),
        })

    print(f"ep {episode+1}/{n_episodes} | updates: {len(actor_losses)}")

print(f"Training hidden states shape: {hidden_np_train.shape}")

## 4. Loss

In [ ]:
def smooth(x, w):
    x = np.array(x, dtype=float)
    if len(x) < w:
        return x, np.arange(len(x))
    return np.convolve(x, np.ones(w) / w, mode="valid"), np.arange(w - 1, len(x))

upd_w = max(1, min(50, len(actor_losses) // 10))

fig, ax = plt.subplots(figsize=(6, 3.5))
s_a, x_a = smooth(actor_losses,  upd_w)
s_c, x_c = smooth(critic_losses, upd_w)
ax.plot(x_a, s_a, label="actor")
ax.plot(x_c, s_c, label="critic", linestyle="--")
ax.set_title("Loss (per gradient update)")
ax.set_xlabel("Update step")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Per-stimulus traces

In [ ]:
td = np.array([
    (d["global_trial"], d["context"], d["stimulus"],
     d["reward_available"], d["licked"], d["value_estimate"], d["lick_count"])
    for d in all_trial_data
])
# cols: 0=global_trial 1=context 2=stimulus 3=reward_available
#       4=licked(1st rew step) 5=value_estimate 6=lick_count

# context block boundaries
_ctx_arr         = np.array([t["context"] for t in state_seq.trial_structure])
_chg_idx         = np.where(np.diff(_ctx_arr) != 0)[0] + 1
ctx_block_starts = np.concatenate([[0], _chg_idx])
ctx_block_ends   = np.concatenate([_chg_idx, [n_trials]])
ctx_block_ctxs   = _ctx_arr[ctx_block_starts]

ctx_colors  = [plt.cm.Set2(v) for v in np.linspace(0, 0.75, max(n_contexts, 1))]
stim_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

smooth_w = max(1, min(20, n_trials // 10))

fig, axes = plt.subplots(4, 1, figsize=(15, 11),
                         gridspec_kw={"height_ratios": [3, 2.5, 2.5, 1], "hspace": 0.45})

# context background shading on data rows only
for bs, be, ctx in zip(ctx_block_starts, ctx_block_ends, ctx_block_ctxs):
    for ax in axes[:3]:
        ax.axvspan(bs - 0.5, be - 0.5, alpha=0.15, color=ctx_colors[ctx], zorder=0, linewidth=0)

for si in range(n_stimuli):
    mask        = td[:, 2] == si
    gt          = td[mask, 0].astype(int)
    val         = td[mask, 5]
    lick        = td[mask, 4]
    lick_rate_w = td[mask, 6] / reward_timesteps

    c = stim_colors[si % len(stim_colors)]
    if smooth_w > 1:
        sv  = np.convolve(val,         np.ones(smooth_w) / smooth_w, mode="valid")
        sl  = np.convolve(lick,        np.ones(smooth_w) / smooth_w, mode="valid")
        slr = np.convolve(lick_rate_w, np.ones(smooth_w) / smooth_w, mode="valid")
        xax = gt[smooth_w - 1:]
    else:
        sv, sl, slr, xax = val, lick, lick_rate_w, gt

    axes[0].plot(xax, sv,  color=c, linewidth=1.2, label=stimuli[si])
    axes[1].plot(xax, sl,  color=c, linewidth=1.2)
    axes[2].plot(xax, slr, color=c, linewidth=1.2)

# ground-truth value lines per context (dotted, same stim colour, faint)
for si in range(n_stimuli):
    c = stim_colors[si % len(stim_colors)]
    for ci in range(n_contexts):
        axes[0].axhline(value_matrix[si, ci], color=c, linestyle=":", linewidth=0.8, alpha=0.5)

stim_handles = [Line2D([0],[0], color=stim_colors[si % len(stim_colors)], label=stimuli[si])
                for si in range(n_stimuli)]
ctx_handles  = [Patch(facecolor=ctx_colors[c], alpha=0.6, label=contexts[c])
                for c in range(n_contexts)]

axes[0].legend(handles=stim_handles + ctx_handles, ncol=n_stimuli + n_contexts,
               fontsize=8, loc="upper left")
axes[0].set_ylabel("Value estimate")
axes[0].set_title("Per-stimulus value estimate  (dotted = ground truth per context)")

axes[1].legend(handles=stim_handles, ncol=n_stimuli, fontsize=8, loc="upper left")
axes[1].set_ylabel("Lick prob.\n(1st rew. step)")
axes[1].set_ylim(-0.02, 1.02)
axes[1].set_title("Lick probability at first reward window timestep")

axes[2].set_ylabel("Lick rate\n(all rew. steps)")
axes[2].set_ylim(-0.02, 1.02)
axes[2].set_xlabel("Global trial index")
axes[2].set_title(f"Mean lick rate across all reward window timesteps ({reward_timesteps} ts)")

# ── trial structure schematic ──────────────────────────────────────────────
ax = axes[3]
avg_iti   = state_seq.iti_durations.mean()
total_ts  = avg_iti + stim_timesteps + reward_timesteps
iti_frac  = avg_iti / total_ts
stim_frac = stim_timesteps / total_ts

for xstart, xend, color, label in [
    (0,                    iti_frac,             "lightgray", f"ITI (~{avg_iti:.0f} ts, jittered)"),
    (iti_frac,             iti_frac + stim_frac, "#4C72B0",   f"Stim ({stim_timesteps} ts)"),
    (iti_frac + stim_frac, 1.0,                  "#DD8452",   f"Reward window ({reward_timesteps} ts)"),
]:
    ax.barh(0, xend - xstart, left=xstart, height=0.5, color=color)
    ax.text((xstart + xend) / 2, 0, label, ha="center", va="center", fontsize=8,
            color="black" if color == "lightgray" else "white")

lick_x = iti_frac + stim_frac
ax.annotate("lick sampled\n(row 2 above)", xy=(lick_x, 0.25), xytext=(lick_x + 0.07, 1.0),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.0), fontsize=7, ha="center")

ax.set_xlim(0, 1)
ax.set_ylim(-0.6, 1.6)
ax.axis("off")
ax.set_title(f"Trial structure (mean total ≈ {total_ts:.0f} ts)", fontsize=8, loc="left", pad=2)

plt.show()

In [ ]:
from scipy.stats import spearmanr
from matplotlib.lines import Line2D

# ── performance metrics ────────────────────────────────────────────────────
licked_arr     = np.array([d["licked"]           for d in all_trial_data])
rew_avail      = np.array([d["reward_available"]  for d in all_trial_data]).astype(bool)
stim_idx_arr   = np.array([d["stimulus"]          for d in all_trial_data])
ctx_idx_arr    = np.array([d["context"]           for d in all_trial_data])
lick_count_arr = np.array([d["lick_count"]        for d in all_trial_data])

pct_reward_consumed = licked_arr[rew_avail].mean()  * 100 if  rew_avail.any() else np.nan
false_alarm_rate    = licked_arr[~rew_avail].mean() * 100 if (~rew_avail).any() else np.nan
lick_rate_window    = lick_count_arr.mean() / reward_timesteps * 100

stim_colors_p = plt.rcParams["axes.prop_cycle"].by_key()["color"]
ctx_markers   = ["o", "s", "^"]

lick_probs_flat, reward_probs_flat = [], []

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax_sc = axes[0]

for si in range(n_stimuli):
    for ci in range(n_contexts):
        mask_sc = (stim_idx_arr == si) & (ctx_idx_arr == ci)
        if mask_sc.sum() == 0:
            continue
        lp = licked_arr[mask_sc].mean()
        rp = float(value_matrix[si, ci])
        lick_probs_flat.append(lp)
        reward_probs_flat.append(rp)
        ax_sc.scatter(rp, lp,
                      color=stim_colors_p[si % len(stim_colors_p)],
                      marker=ctx_markers[ci % len(ctx_markers)],
                      s=80, zorder=3)
        ax_sc.annotate(f"{stimuli[si]},{contexts[ci]}", (rp, lp),
                       textcoords="offset points", xytext=(5, 3), fontsize=7)

lick_probs_flat   = np.array(lick_probs_flat)
reward_probs_flat = np.array(reward_probs_flat)
r_sp = spearmanr(reward_probs_flat, lick_probs_flat)[0] if len(lick_probs_flat) > 1 else np.nan

ax_sc.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5, label="ideal")
ax_sc.set_xlim(-0.05, 1.05); ax_sc.set_ylim(-0.05, 1.05)
ax_sc.set_xlabel("Reward probability"); ax_sc.set_ylabel("Lick probability")
ax_sc.set_title(f"Lick–value calibration  (Spearman r = {r_sp:.3f})")

stim_h = [Line2D([0],[0], color=stim_colors_p[si % len(stim_colors_p)],
                 marker="o", linestyle="none", label=stimuli[si]) for si in range(n_stimuli)]
ctx_h  = [Line2D([0],[0], color="gray",
                 marker=ctx_markers[ci % len(ctx_markers)], linestyle="none",
                 label=contexts[ci]) for ci in range(n_contexts)]
ax_sc.legend(handles=stim_h + ctx_h, fontsize=8, ncol=2, loc="upper left")

ax_bar = axes[1]
bars = ax_bar.bar(
    ["Reward\nconsumed (%)", "False alarm\nrate (%)", "Lick rate\nin window (%)"],
    [pct_reward_consumed, false_alarm_rate, lick_rate_window],
    color=["steelblue", "salmon", "gray"], width=0.5,
)
ax_bar.set_ylim(0, 110); ax_bar.set_ylabel("%")
ax_bar.set_title("Performance summary")
for bar, val in zip(bars, [pct_reward_consumed, false_alarm_rate, lick_rate_window]):
    if not np.isnan(val):
        ax_bar.text(bar.get_x() + bar.get_width() / 2, val + 1,
                    f"{val:.1f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

## 6. Post-training activations

In [ ]:
def t2np(tensor):
    return np.array(tensor.detach().cpu().tolist(), dtype=np.float32)

actor_critic.eval()
obs, _ = env.reset()
agent.reset()

obs_seq, action_seq = [], []
done = False
while not done:
    obs_seq.append(obs.copy())
    action, _, _ = agent.act(obs)
    action_seq.append(action)
    obs, _, done, _, _ = env.step(action)

with torch.no_grad():
    obs_t = torch.FloatTensor(np.stack(obs_seq)).unsqueeze(0).to(device)
    _, _, hidden_all = actor_critic(obs_t)
hidden_np = t2np(hidden_all.squeeze(0))   # (T, hidden_size)

# trial metadata
context_arr          = np.array([t["context"]          for t in state_seq.trial_structure])
stimulus_arr         = np.array([t["stimulus"]         for t in state_seq.trial_structure])
reward_available_arr = np.array([t["reward_available"] for t in state_seq.trial_structure])

_cc = defaultdict(int)
phase_arr = np.zeros(n_trials, dtype=int)
for i, ctx in enumerate(context_arr):
    phase_arr[i] = _cc[ctx] // trials_per_phase
    _cc[ctx] += 1

action_arr   = np.array([action_seq[t["reward_window"][0]] for t in state_seq.trial_structure])
rewarded_arr = (action_arr == TaskEnv.LICK) & reward_available_arr

stim_hidden   = np.stack([hidden_np[t["stim_window"][0]   : t["stim_window"][1]]   for t in state_seq.trial_structure])
reward_hidden = np.stack([hidden_np[t["reward_window"][0] : t["reward_window"][1]] for t in state_seq.trial_structure])

activations = {
    "hidden_states":    hidden_np,
    "stim_hidden":      stim_hidden,
    "reward_hidden":    reward_hidden,
    "context":          context_arr,
    "stimulus":         stimulus_arr,
    "phase":            phase_arr,
    "reward_available": reward_available_arr,
    "action":           action_arr,
    "rewarded":         rewarded_arr,
    "trial_structure":  state_seq.trial_structure,
}
print(f"hidden_states : {hidden_np.shape}")
print(f"stim_hidden   : {stim_hidden.shape}")
print(f"phases present: {np.unique(phase_arr)}")

In [ ]:
# activation heatmaps — mean hidden state per epoch, per stimulus
iti_hidden_mean = np.array([
    hidden_np[t["iti_window"][0]:t["iti_window"][1]].mean(0)
    if t["iti_window"][1] > t["iti_window"][0] else np.zeros(hidden_np.shape[1])
    for t in state_seq.trial_structure
])   # (n_trials, H)

stim_hidden_mean   = activations["stim_hidden"].mean(1)    # (n_trials, H)
reward_hidden_mean = activations["reward_hidden"].mean(1)  # (n_trials, H)

epochs_data = [
    ("ITI",    iti_hidden_mean),
    ("Stim",   stim_hidden_mean),
    ("Reward", reward_hidden_mean),
]

all_vals = np.concatenate([d.ravel() for _, d in epochs_data])
vmin_act, vmax_act = np.percentile(all_vals, 2), np.percentile(all_vals, 98)

fig, axes = plt.subplots(n_stimuli, 3, figsize=(12, 2.5 * n_stimuli), sharey=True)

for si in range(n_stimuli):
    mask     = stimulus_arr == si
    ctx_this = context_arr[mask]

    chg        = np.where(np.diff(ctx_this) != 0)[0] + 1 if ctx_this.size > 1 else np.array([], dtype=int)
    blk_starts = np.concatenate([[0], chg])
    blk_ends   = np.concatenate([chg, [mask.sum()]])
    blk_ctxs   = ctx_this[blk_starts]

    for col, (epoch_name, epoch_data) in enumerate(epochs_data):
        ax   = axes[si, col]
        data = epoch_data[mask]

        im = ax.imshow(data.T, aspect="auto", cmap="viridis",
                       vmin=vmin_act, vmax=vmax_act, interpolation="nearest")

        for bs, be, ctx in zip(blk_starts, blk_ends, blk_ctxs):
            ax.axvspan(bs - 0.5, be - 0.5, alpha=0.2,
                       color=ctx_colors[ctx], zorder=2, linewidth=0)

        if si == 0:
            ax.set_title(epoch_name, fontsize=10)
        if col == 0:
            ax.set_ylabel(f"{stimuli[si]}\nUnit", fontsize=8)
        if si == n_stimuli - 1:
            ax.set_xlabel("Trial", fontsize=8)
        ax.tick_params(labelsize=7)

fig.colorbar(im, ax=axes, shrink=0.5, label="Activation", pad=0.02)

ctx_handles = [Patch(facecolor=ctx_colors[c], alpha=0.7, label=contexts[c])
               for c in range(n_contexts)]
axes[0, -1].legend(handles=ctx_handles, loc="upper right", fontsize=7, framealpha=0.7)

plt.suptitle("Mean hidden-state activations per period (rows = stimuli)", y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

## 7. Decoding — post-training

In [ ]:
pooling = "average"
n_folds = 5

# stimulus generalisation
stim_acc       = pairwise_decode(activations, period="stim", pooling=pooling, n_folds=n_folds)
cross_stim_acc = crosscontext_decode(activations, period="stim", pooling=pooling)
gen_stim       = generalisation_matrix(stim_acc, cross_stim_acc)

# value decoding
r_within = value_decode_within(activations, period="stim", pooling=pooling,
                               value_matrix=value_matrix, n_folds=n_folds)
r_cross  = value_decode_cross(activations,  period="stim", pooling=pooling,
                              value_matrix=value_matrix)
gm_val   = value_gen_matrix(r_within, r_cross)

cell_size = 1.1
fig, axes = plt.subplots(1, 2,
                         figsize=(cell_size * n_contexts * 2 + 1, cell_size * n_contexts),
                         squeeze=False)
plot_generalisation_heatmap(axes[0, 0], gen_stim, contexts,
                            vmin=0, vmax=1, cmap="seismic",
                            colorbar_label="Accuracy",
                            title="Stim generalisation  (white = chance)")
plot_generalisation_heatmap(axes[0, 1], gm_val, contexts,
                            vmin=-1, vmax=1, cmap="seismic",
                            colorbar_label="Pearson r",
                            title="Value decoding  (white = r=0)")
fig.suptitle(f"Post-training decoding  ·  stim period  ·  {pooling} pooling", y=1.02)
plt.tight_layout()
plt.show()

## 8. Decoding — per context repetition (post-training activations)

In [ ]:
n_phases = int(activations["phase"].max()) + 1
phases   = np.arange(n_phases)

within_stim_ph = np.full((n_phases, n_contexts), np.nan)
cross_stim_ph  = np.full((n_phases, n_contexts, n_contexts), np.nan)
within_val_ph  = np.full((n_phases, n_contexts), np.nan)
cross_val_ph   = np.full((n_phases, n_contexts, n_contexts), np.nan)

for p in range(n_phases):
    print(f"phase {p+1}/{n_phases}", end="\r")
    act_p = filter_act_dict(activations, activations["phase"] == p)

    acc_p = pairwise_decode(act_p, period="stim", pooling=pooling, n_folds=n_folds)
    for c in range(n_contexts):
        within_stim_ph[p, c] = mean_pairs(acc_p[c])

    cx_p = crosscontext_decode(act_p, period="stim", pooling=pooling)
    for ct in range(n_contexts):
        for ce in range(n_contexts):
            if ct != ce:
                cross_stim_ph[p, ct, ce] = mean_pairs(cx_p[ct, ce])

    within_val_ph[p] = value_decode_within(act_p, period="stim", pooling=pooling,
                                           value_matrix=value_matrix, n_folds=n_folds)
    cross_val_ph[p]  = value_decode_cross(act_p, period="stim", pooling=pooling,
                                          value_matrix=value_matrix)
print("done")

mean_cross_stim_ph = np.array([mean_offdiag(cross_stim_ph[p]) for p in range(n_phases)])
mean_cross_val_ph  = np.array([mean_offdiag(cross_val_ph[p])  for p in range(n_phases)])

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)

for c in range(n_contexts):
    axes[0, 0].plot(phases, within_stim_ph[:, c], color=ctx_colors[c], label=contexts[c], marker="o", ms=4)
axes[0, 0].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[0, 0].set_ylabel("Mean pairwise accuracy")
axes[0, 0].set_title("Stimulus decoding — within context")
axes[0, 0].legend(fontsize=8)
axes[0, 0].set_ylim(0, 1)

for ct in range(n_contexts):
    for ce in range(n_contexts):
        if ct != ce:
            axes[0, 1].plot(phases, cross_stim_ph[:, ct, ce], color="gray", alpha=0.3, linewidth=0.8)
axes[0, 1].plot(phases, mean_cross_stim_ph, color="black", linewidth=1.5, marker="o", ms=4, label="mean")
axes[0, 1].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[0, 1].set_title("Stimulus decoding — cross-context")
axes[0, 1].set_ylim(0, 1)
axes[0, 1].legend(fontsize=8)

for c in range(n_contexts):
    axes[1, 0].plot(phases, within_val_ph[:, c], color=ctx_colors[c], label=contexts[c], marker="o", ms=4)
axes[1, 0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1, 0].set_ylabel("Pearson r")
axes[1, 0].set_xlabel("Context repetition")
axes[1, 0].set_title("Value decoding — within context")
axes[1, 0].set_ylim(-1, 1)

for ct in range(n_contexts):
    for ce in range(n_contexts):
        if ct != ce:
            axes[1, 1].plot(phases, cross_val_ph[:, ct, ce], color="gray", alpha=0.3, linewidth=0.8)
axes[1, 1].plot(phases, mean_cross_val_ph, color="black", linewidth=1.5, marker="o", ms=4, label="mean")
axes[1, 1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1, 1].set_xlabel("Context repetition")
axes[1, 1].set_title("Value decoding — cross-context")
axes[1, 1].set_ylim(-1, 1)
axes[1, 1].legend(fontsize=8)

for ax in axes.flat:
    ax.set_xticks(phases)

fig.suptitle(f"Decoding by context repetition  ·  stim period  ·  {pooling} pooling  (post-training)", y=1.01)
plt.tight_layout()
plt.show()

## 9. Decoding — per context repetition (training-time activations)

In [ ]:
stim_hidden_tr   = np.stack([hidden_np_train[t["stim_window"][0]   : t["stim_window"][1]]   for t in state_seq.trial_structure])
reward_hidden_tr = np.stack([hidden_np_train[t["reward_window"][0] : t["reward_window"][1]] for t in state_seq.trial_structure])

_cc_tr = defaultdict(int)
phase_arr_tr = np.zeros(n_trials, dtype=int)
for i, ctx in enumerate(context_arr):
    phase_arr_tr[i] = _cc_tr[ctx] // trials_per_phase
    _cc_tr[ctx] += 1

activations_train = {
    "hidden_states":    hidden_np_train,
    "stim_hidden":      stim_hidden_tr,
    "reward_hidden":    reward_hidden_tr,
    "context":          context_arr,
    "stimulus":         stimulus_arr,
    "phase":            phase_arr_tr,
    "reward_available": reward_available_arr,
    "trial_structure":  state_seq.trial_structure,
}

n_phases_tr = int(phase_arr_tr.max()) + 1
cell_size   = 1.1

fig, axes = plt.subplots(
    2, n_phases_tr,
    figsize=(cell_size * n_contexts * n_phases_tr + 1, cell_size * n_contexts * 2 + 1),
    squeeze=False,
)

for p in range(n_phases_tr):
    print(f"  phase {p+1}/{n_phases_tr}", end="\r")
    act_p = filter_act_dict(activations_train, phase_arr_tr == p)

    acc_p  = pairwise_decode(act_p, period="stim", pooling=pooling, n_folds=n_folds)
    cx_p   = crosscontext_decode(act_p, period="stim", pooling=pooling)
    gm_s_p = generalisation_matrix(acc_p, cx_p)

    r_win_p = value_decode_within(act_p, period="stim", pooling=pooling,
                                  value_matrix=value_matrix, n_folds=n_folds)
    r_crs_p = value_decode_cross(act_p, period="stim", pooling=pooling,
                                 value_matrix=value_matrix)
    gm_v_p  = value_gen_matrix(r_win_p, r_crs_p)

    for row, (gm, vmin, vmax, clabel, ylabel) in enumerate([
        (gm_s_p, 0,  1,  "Accuracy",  "Stim\nTrain ctx"),
        (gm_v_p, -1, 1,  "Pearson r", "Value\nTrain ctx"),
    ]):
        plot_generalisation_heatmap(
            axes[row, p], gm, contexts,
            vmin=vmin, vmax=vmax, cmap="seismic",
            colorbar_label=clabel,
            title=f"Rep {p+1}" if row == 0 else None,
            ylabel=ylabel if p == 0 else "Train ctx",
        )

print("done")
fig.suptitle(
    f"Training-time decoding by context repetition  ·  stim period  ·  {pooling} pooling\n"
    f"Row 1: stim generalisation (white = chance)  ·  Row 2: value decoding (white = r=0)",
    y=1.02, fontsize=10,
)
plt.tight_layout()
plt.show()

## 10. Save

In [ ]:
run_id = (
    f"RNN_3s2c_swap"
    f"_h{hidden_size}_g{recurrent_gain}"
    f"_ep{n_episodes}_upd{update_every}"
    f"_seed{seed}"
    f"_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)

run_dir = Path("../results") / run_id
run_dir.mkdir(parents=True, exist_ok=True)

with open(run_dir / "activations.pkl", "wb") as f:
    pickle.dump(activations, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(run_dir / "activations_train.pkl", "wb") as f:
    pickle.dump(activations_train, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"run_id : {run_id}")
print(f"Saved  → {run_dir}")